In [ ]:
import os
import shutil
import pyNN.spiNNaker as sim
import torch
import logging
from datetime import datetime

logging.getLogger().setLevel(logging.ERROR)
for logger_name in list(logging.root.manager.loggerDict.keys()):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

def force_spynnaker_config():
    """Automatically generate the configuration file to force energy reporting."""
    config_content = "[Reports]\nwrite_energy_report = True\n"
    with open(".spynnaker.cfg", "w") as f:
        f.write(config_content)
    with open("spynnaker.cfg", "w") as f:
        f.write(config_content)
    print("[*] Configuration forced: Energy report enabled.")

force_spynnaker_config()

DATASET = "nmnist"  # Choices: "nmnist", "cifar10_dvs", "dvs_gesture", "nepic_kitchens"
SIMULATION_TIME_MS = 20.0

WEIGHTS_PATHS = {
    "nmnist": "networks/nmnist_best.pth",
    "cifar10_dvs": "networks/cifar10_dvs_best.pth",
    "dvs_gesture": "networks/dvs_gesture_best.pth",
    "nepic_kitchens": "networks/nepic_kitchens_best.pth",
}

TOPOLOGIES = {
    "nmnist": [2312, 256, 10],
    "cifar10_dvs": [1568, 50176, 25088, 12544, 4608, 10],
    "dvs_gesture": [32768, 131072, 65536, 32768, 16384, 11],
    "nepic_kitchens": [116736, 262144, 131072, 65536, 32768, 16384, 8192, 4096, 8]
}

def format_weights(weight_matrix):
    """
    Transforms a PyTorch weight matrix into pyNN.spiNNaker compatible connectors.
    """
    connector_list = []
    if len(weight_matrix.shape) > 2:
        weight_matrix = weight_matrix.view(weight_matrix.size(0), -1)
        
    for i in range(weight_matrix.shape[1]):
        for j in range(weight_matrix.shape[0]):
            connector_list.append((i, j, float(weight_matrix[j, i]), 1.0))
    return connector_list

def organize_reports(dataset_name, mode="inference"):
    """
    Moves the raw SpiNNaker report into reports/<mode>/<dataset_name>/.
    """
    report_dir = "reports"
    if not os.path.exists(report_dir):
        return
        
    subdirs = [os.path.join(report_dir, d) for d in os.listdir(report_dir) 
               if os.path.isdir(os.path.join(report_dir, d)) and d.startswith("20")]
              
    if not subdirs:
        return
        
    latest_report = max(subdirs, key=os.path.getmtime)
    folder_name = os.path.basename(latest_report)
    
    target_dir = os.path.join(report_dir, mode, dataset_name)
    os.makedirs(target_dir, exist_ok=True)
    
    target_path = os.path.join(target_dir, folder_name)
    if not os.path.exists(target_path):
        shutil.copytree(latest_report, target_path)
        shutil.rmtree(latest_report)
        print(f"\n[*] Energy report successfully sorted in: {target_path}")

def deploy_spinnaker_inference(dataset_name, pth_path, simulation_time):
    print(f"=== SpiNNaker Inference Deployment: {dataset_name.upper()} ===")
    
    if dataset_name not in TOPOLOGIES:
        raise ValueError(f"Unknown dataset: {dataset_name}")
        
    layers_sizes = TOPOLOGIES[dataset_name]
    
    if not os.path.exists(pth_path):
        raise FileNotFoundError(f"Weights file not found: {pth_path}")
        
    print(f"[*] Loading weights from: {pth_path}")
    state_dict = torch.load(pth_path, map_location="cpu")
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
    
    sim.setup(timestep=1.0)

    print("[*] Configuring network on SpiNNaker hardware...")
    input_pop = sim.Population(
        layers_sizes[0], 
        sim.SpikeSourcePoisson(rate=50.0, duration=simulation_time),
        label="Input_Layer"
    )
    
    populations = [input_pop]
    
    for i, size in enumerate(layers_sizes[1:]):
        pop = sim.Population(size, sim.IF_curr_exp(), label=f"Layer_{i+1}")
        populations.append(pop)
        
    output_pop = populations[-1]
    output_pop.record(["spikes"])
    
    print("[*] Routing synaptic connections (this may take a few minutes)...")
    
    try:
        weight_keys = [k for k in state_dict.keys() if "weight" in k and "bn" not in k.lower()]
        
        for idx in range(len(populations) - 1):
            if idx < len(weight_keys):
                w_matrix = state_dict[weight_keys[idx]].cpu()
                connectors = format_weights(w_matrix)
                sim.Projection(
                    populations[idx], 
                    populations[idx+1], 
                    sim.FromListConnector(connectors)
                )
            else:
                sim.Projection(populations[idx], populations[idx+1], sim.OneToOneConnector(weight=0.5))
    except Exception as e:
        print(f"[!] Warning during weight routing: {e}")
        print("[!] Using default connectors for the test.")
        for idx in range(len(populations) - 1):
            sim.Projection(populations[idx], populations[idx+1], sim.OneToOneConnector(weight=0.5))

    print(f"[*] Launching simulation ({simulation_time} ms)...")
    sim.run(simulation_time)
    
    spikes = output_pop.get_data("spikes")
    sim.end()
    
    organize_reports(dataset_name, mode="inference")
    
    return spikes

if __name__ == "__main__":
    pth_file = WEIGHTS_PATHS.get(DATASET, "")
    
    start_time = datetime.now()
    try:
        results = deploy_spinnaker_inference(DATASET, pth_file, SIMULATION_TIME_MS)
        
        print("\n==================================================")
        print(f"INFERENCE RESULTS: {DATASET.upper()}")
        print("==================================================")
        print("Inference completed successfully.")
        print(f"Total processing time: {datetime.now() - start_time}")
        print("Output layer activity (spikes):")
        print(results.segments[0].spiketrains)
        
    except Exception as e:
        print(f"\n[X] Critical error during SpiNNaker execution: {e}")